# Stage 4 — Phase 5.2/5.3: Fine-Tuning Timing Probe + Training

Uses `src/stage4_symbol_detection/train.py`. Starts with **Qwen3-VL** to prove the
pipeline end-to-end (measured timing probe, not the hand-estimate in
Stage4_Checklist_Status.md), then repeats for InternVL3 and Molmo2-O-7B.

Per the 2026-07-10 resequencing decision: this trains a SINGLE combined LoRA
directly on the detection task (5.2+5.3 merged), not a separate domain-adapt-then-
adapter sequence. The proper task-neutral domain-adapt pass is deferred to
end-of-Stage-4, as its own separate checkpoint.

## 1. Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Pull latest code (run this cell any time after a fix, no other steps needed)

In [ ]:
import os
if os.path.exists('/content/pid-ml'):
    !cd /content/pid-ml && git pull
else:
    !git clone https://github.com/TomGeorge45/pid-opensrc-substitution.git /content/pid-ml

## 3. Install dependencies

`peft` is new — needed for LoRA. Using the transformers version already verified working
for Qwen3-VL/InternVL3 (5.x line) for this first probe.

In [3]:
!pip install -q transformers accelerate peft pycocotools supervision \
    kagglehub kaggle qwen-vl-utils einops timm

## 4. Import (always re-run this after pulling — it force-reloads, safe to run repeatedly)

In [ ]:
import sys, importlib
if '/content/pid-ml/src' not in sys.path:
    sys.path.insert(0, '/content/pid-ml/src')

import stage4_symbol_detection.qwen_candidate as qwen_candidate
import stage4_symbol_detection.train as train
importlib.reload(qwen_candidate)
importlib.reload(train)
print("Modules loaded/reloaded.")

## 5. Load Qwen3-VL and build training examples

In [ ]:
processor, model = qwen_candidate.load()

examples = train.build_examples("qwen3vl")
print(f"Total training examples: {len(examples)}")
n_gupta = sum(1 for e in examples if e['source'] == 'gupta')
n_kaggle = sum(1 for e in examples if e['source'] == 'kaggle')
print(f"  gupta tiles: {n_gupta} | kaggle images: {n_kaggle}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/2.72G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

Loaded Qwen/Qwen3-VL-8B-Instruct in 63.2s
VRAM used: 17.5 GB


## 6. Apply LoRA

In [ ]:
model = train.setup_lora(model)

## 7. Timing probe — MEASURED, not estimated

This is the number that actually matters for planning — everything in
Stage4_Checklist_Status.md before this point is a hand-wavy guess.

In [ ]:
avg_step_time = train.time_training_steps(
    processor, model, qwen_candidate.SYMBOL_DETECTION_PROMPT, examples,
    n_steps=5, image_first=True,
)

n_examples = len(examples)
for n_epochs in (1, 2, 3):
    total_s = avg_step_time * n_examples * n_epochs
    print(f"{n_epochs} epoch(s): {total_s/3600:.2f}h projected (measured {avg_step_time:.2f}s/step x {n_examples} examples)")